In [ ]:
import os
import random
from pathlib import Path

missing_modules = []

try:
    import numpy as np
except ModuleNotFoundError:
    np = None
    missing_modules.append("numpy")

try:
    import rasterio
    from rasterio.windows import Window
except ModuleNotFoundError:
    rasterio = None
    Window = None
    missing_modules.append("rasterio")

try:
    import torch
    import torch.nn as nn
    from torch.utils.data import Dataset, DataLoader
except ModuleNotFoundError:
    torch = None
    nn = None
    Dataset = None
    DataLoader = None
    missing_modules.append("torch")


# =========================
# 1. ??????
# =========================

image_dir = os.getenv("IMAGE_DIR")
label_dir = os.getenv("LABEL_DIR")

patch_size = 128
stride = 128

batch_size = 16
num_epochs = 15
learning_rate = 1e-3

min_positive_pixels = 10
negative_keep_ratio = 0.3
max_patches_per_image = 300

train_ratio = 0.7
val_ratio = 0.15
test_ratio = 0.15

random_seed = 42
random.seed(random_seed)

if np is not None:
    np.random.seed(random_seed)

if torch is not None:
    torch.manual_seed(random_seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(random_seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
else:
    device = "cpu"

print("????:", device)

if missing_modules:
    print("??????:", ", ".join(missing_modules))
    print("????: pip install numpy rasterio torch")


# =========================
# 2. ?? image ? label
# =========================

def find_data_dir(dir_name):
    candidates = [
        Path.cwd(),
        Path.cwd() / "data",
        Path.home() / "Desktop",
        Path.home() / "Downloads",
        Path.home() / "Documents",
    ]

    for base in candidates:
        candidate = base / dir_name
        if candidate.exists():
            return candidate

    return None

if image_dir:
    image_dir = Path(image_dir)
else:
    image_dir = find_data_dir("image_6band")

if label_dir:
    label_dir = Path(label_dir)
else:
    label_dir = find_data_dir("label_6band")

print("image_dir:", image_dir)
print("label_dir:", label_dir)

pairs = []

if image_dir is not None and label_dir is not None and image_dir.exists() and label_dir.exists():
    image_files = sorted(image_dir.glob("*.tif"))
    label_files = sorted(label_dir.glob("*.tif"))
    label_dict = {p.stem: p for p in label_files}

    for img_path in image_files:
        if img_path.stem in label_dict:
            pairs.append((img_path, label_dict[img_path.stem]))
else:
    print("??? image_6band / label_6band ??????? image_dir?label_dir ??????? IMAGE_DIR?LABEL_DIR?")

print("???? image-label ??:", len(pairs))

if len(pairs) == 0:
    print("????????????? cell ?????????????????????")


# =========================
# 3. ?????? train / val / test
# =========================

random.shuffle(pairs)

n_total = len(pairs)
n_train = int(n_total * train_ratio)
n_val = int(n_total * val_ratio)

train_pairs = pairs[:n_train]
val_pairs = pairs[n_train:n_train + n_val]
test_pairs = pairs[n_train + n_val:]

print("??????:", len(train_pairs))
print("??????:", len(val_pairs))
print("??????:", len(test_pairs))


In [ ]:
epochs = range(1, len(history["train_loss"]) + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(epochs, history["train_loss"], marker="o", label="Train Loss")
axes[0].plot(epochs, history["val_loss"], marker="o", label="Val Loss")
axes[0].set_title("Loss Curve")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].plot(epochs, history["train_acc"], marker="o", label="Train Acc")
axes[1].plot(epochs, history["val_acc"], marker="o", label="Val Acc")
axes[1].plot(epochs, history["val_f1"], marker="o", label="Val F1")
axes[1].set_title("Accuracy / F1 Curve")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Score")
axes[1].set_ylim(0, 1.05)
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()


In [ ]:
# Comparison: model prediction / NBR weak label / raw image
from pathlib import Path
import importlib.util

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn

image_dir = Path(r"E:\卫星遥感数据\02草原火灾\image_6band_不含烟雾")
label_dir = Path(r"E:\卫星遥感数据\02草原火灾\label_6band_不含烟雾")
patch_size = 128
stride = 128
best_model_path = Path(r"C:\Users\lenovo\Desktop\small_fire_classifier.pth")
if not best_model_path.exists():
    best_model_path = Path("small_fire_classifier.pth")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
HAS_RASTERIO = importlib.util.find_spec("rasterio") is not None
HAS_TIFFFILE = importlib.util.find_spec("tifffile") is not None

if HAS_RASTERIO:
    import rasterio
elif HAS_TIFFFILE:
    import tifffile
else:
    raise ImportError("Need rasterio or tifffile to read tif images.")

def read_full_image(path: Path):
    if HAS_RASTERIO:
        with rasterio.open(path) as src:
            return src.read()
    arr = tifffile.imread(path)
    if arr.ndim == 2:
        arr = arr[np.newaxis, ...]
    elif arr.ndim == 3 and arr.shape[0] > arr.shape[-1]:
        arr = np.moveaxis(arr, -1, 0)
    return arr

def read_label(path: Path):
    label = read_full_image(path)
    if label.ndim == 3:
        label = label[0]
    return label

image_files = sorted(image_dir.glob("*.tif"))
label_files = sorted(label_dir.glob("*.tif"))
label_dict = {p.stem: p for p in label_files}
pairs = [(img_path, label_dict[img_path.stem]) for img_path in image_files if img_path.stem in label_dict]

if not pairs:
    raise RuntimeError(f"No matching image/label pairs found under {image_dir} and {label_dir}.")
if not best_model_path.exists():
    raise FileNotFoundError(f"Model file not found: {best_model_path}")

class SmallCNN(nn.Module):
    def __init__(self, in_channels, num_classes=2):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

compare_image_index = 0
if not (0 <= compare_image_index < len(pairs)):
    raise IndexError(f"compare_image_index={compare_image_index} out of range, total pairs={len(pairs)}")

img_path, lab_path = pairs[compare_image_index]
full_img = read_full_image(img_path).astype(np.float32)
label_img = read_label(lab_path)

if full_img.ndim != 3:
    raise ValueError(f"Expected image shape [C, H, W], got {full_img.shape}")
if label_img.ndim != 2:
    raise ValueError(f"Expected label shape [H, W], got {label_img.shape}")

channels, height, width = full_img.shape
if label_img.shape != (height, width):
    raise ValueError(f"Image/label size mismatch: image={(height, width)}, label={label_img.shape}")
if height < patch_size or width < patch_size:
    raise ValueError(f"Image size {(height, width)} is smaller than patch_size={patch_size}")

model = SmallCNN(in_channels=channels, num_classes=2).to(device)
state_dict = torch.load(best_model_path, map_location=device)
model.load_state_dict(state_dict)
model.eval()

full_img = np.nan_to_num(full_img, nan=0.0, posinf=0.0, neginf=0.0)
norm_img = np.clip(full_img, 0, 10000) / 10000.0

rgb_band_indices = tuple(range(min(3, channels)))
if channels >= 4:
    rgb_band_indices = (2, 1, 0)

nir_band_index = min(4, channels - 1)
swir_band_index = min(5, channels - 1)
if swir_band_index == nir_band_index and channels >= 2:
    nir_band_index = channels - 2
    swir_band_index = channels - 1

coords = []
patches = []
for y in range(0, height - patch_size + 1, stride):
    for x in range(0, width - patch_size + 1, stride):
        coords.append((x, y))
        patches.append(norm_img[:, y:y + patch_size, x:x + patch_size])

if not patches:
    raise RuntimeError("No valid prediction window was generated.")

batch_size_infer = 128
probs = []
with torch.no_grad():
    for start in range(0, len(patches), batch_size_infer):
        batch = np.stack(patches[start:start + batch_size_infer])
        batch_tensor = torch.tensor(batch, dtype=torch.float32, device=device)
        logits = model(batch_tensor)
        probs.extend(torch.softmax(logits, dim=1)[:, 1].detach().cpu().numpy().tolist())

prob_map = np.zeros((height, width), dtype=np.float32)
count_map = np.zeros((height, width), dtype=np.float32)
for (x, y), prob_fire in zip(coords, probs):
    prob_map[y:y + patch_size, x:x + patch_size] += prob_fire
    count_map[y:y + patch_size, x:x + patch_size] += 1

valid_mask = count_map > 0
pred_map = np.zeros_like(prob_map)
pred_map[valid_mask] = prob_map[valid_mask] / count_map[valid_mask]

nir = norm_img[nir_band_index]
swir = norm_img[swir_band_index]
nbr = (nir - swir) / (nir + swir + 1e-6)
nbr_threshold = 0.10
nbr_weak_label = (nbr < nbr_threshold).astype(np.uint8)

rgb = np.stack([norm_img[i] for i in rgb_band_indices], axis=-1)
rgb = np.clip(rgb, 0, 1)
if rgb.shape[-1] == 1:
    rgb = np.repeat(rgb, 3, axis=-1)
elif rgb.shape[-1] == 2:
    rgb = np.concatenate([rgb, rgb[..., :1]], axis=-1)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
im0 = axes[0].imshow(pred_map, cmap="inferno", vmin=0, vmax=1)
axes[0].set_title("Model Prediction")
plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)
axes[1].imshow(nbr_weak_label, cmap="gray")
axes[1].set_title(f"NBR Weak Label (thr={nbr_threshold})")
axes[2].imshow(rgb)
axes[2].set_title(f"Raw Image RGB {rgb_band_indices}")

label_mask = (label_img > 0).astype(np.uint8)
for ax in axes:
    ax.contour(label_mask, levels=[0.5], colors="cyan", linewidths=0.6)
    ax.set_axis_off()

plt.suptitle(img_path.name)
plt.tight_layout()
plt.show()

print("image:", img_path)
print("label:", lab_path)
print("image shape:", full_img.shape)
print("prediction range:", float(pred_map.min()), float(pred_map.max()))
print("NBR range:", float(nbr.min()), float(nbr.max()))
print("bands used -> rgb:", rgb_band_indices, "nir:", nir_band_index, "swir:", swir_band_index)
